# vLLM on GPU vs TPU — the numbers behind the post

Every figure in the Nov 26 GDE post, recomputed. Prices are published list rates (Google Cloud TPU price list, checked Sep–Nov 2026; H100 via Qovery, Oct 2026). Throughput assumptions in the $/1M-token model are ILLUSTRATIVE — labelled as such — because no controlled GPU-vs-TPU benchmark of the same vLLM version exists yet; the price and memory arithmetic is exact.

In [ ]:
# 1. Cost per 100 peak bf16 TFLOPS, from published list prices
chips = {
    "TPU v5e":          dict(price=1.20, tflops=197,  hbm_gb=16),
    "TPU v6e (Trillium)": dict(price=2.70, tflops=918,  hbm_gb=32),
    "TPU v5p":          dict(price=4.20, tflops=459,  hbm_gb=95),
    "Ironwood (v7x)":   dict(price=12.00, tflops=2307, hbm_gb=192),
    "H100 SXM":         dict(price=6.88, tflops=989,  hbm_gb=80),
}
print(f"{'chip':20s} {'$/chip-hr':>9s} {'$/100 TFLOPS-hr':>16s}")
for name, d in chips.items():
    per100 = d["price"] / d["tflops"] * 100
    d["per100"] = per100
    print(f"{name:20s} {d['price']:9.2f} {per100:16.2f}")
# Trillium is the cheapest peak compute on list; matches the $0.29 figure in the post.
assert abs(chips["TPU v6e (Trillium)"]["per100"] - 0.29) < 0.01
assert abs(chips["TPU v5e"]["per100"] - 0.61) < 0.01
assert chips["TPU v6e (Trillium)"]["per100"] == min(d["per100"] for d in chips.values())

In [ ]:
# 2. Cost per 1M tokens — ILLUSTRATIVE throughput model
# Assumptions (labelled, not measured): decode throughput per chip for an 8B FP8 model,
# 4K context, 128 output tokens. v6e/v5e ratio 1.67x matches the measured Gemma 4 E2B
# band (1.62-1.68x) on workloads that fit in v5e; H100 figure is a placeholder for
# the same class of workload. 8-chip slice, 50% average utilization.
assumed_tps = {"TPU v5e": 900, "TPU v6e (Trillium)": 1500, "H100 SXM": 2200}
chips_n, util = 8, 0.50
print(f"{'slice':22s} {'$/hr':>7s} {'tok/hr':>12s} {'$/1M tokens':>12s}")
for name, tps in assumed_tps.items():
    cost_hr = chips[name]["price"] * chips_n
    tok_hr = tps * chips_n * 3600 * util
    print(f"{name + ' x8':22s} {cost_hr:7.2f} {tok_hr:12,.0f} {cost_hr / tok_hr * 1e6:12.2f}")
# v6e slice: 8*2.70=$21.60/hr, 1500*8*3600*0.5 = 21.6M tok/hr -> $1.00/1M tokens.
assert abs(8 * 2.70 / (1500 * 8 * 3600 * 0.50) * 1e6 - 1.00) < 0.01
# 3-year commitment cuts v6e to $1.22/chip-hr: same slice costs $0.45/1M tokens.
assert abs(8 * 1.22 / (1500 * 8 * 3600 * 0.50) * 1e6 - 0.45) < 0.01

In [ ]:
# 3. KV-cache concurrency: how many 4K-token sequences fit per chip?
# 8B model, FP8 weights = 8 GB. GQA KV per token = 2 (K+V) * layers * kv_heads * head_dim
# bytes at FP8 (1 byte): 2 * 36 * 8 * 128 = 73,728 bytes/token. Reserve 1 GB (v5e/v6e)
# or 2 GB (H100) for activations/fragmentation.
layers, kv_heads, head_dim = 36, 8, 128
kv_per_token = 2 * layers * kv_heads * head_dim  # FP8, 1 byte
print("KV bytes/token:", kv_per_token)
weights_gb, seq_tokens = 8, 4096
for name, reserve_gb in [("TPU v5e", 1), ("TPU v6e (Trillium)", 1), ("H100 SXM", 2)]:
    kv_gb = chips[name]["hbm_gb"] - weights_gb - reserve_gb
    max_tokens = kv_gb * 1e9 / kv_per_token
    print(f"{name:20s} KV headroom {kv_gb:5.0f} GB -> {max_tokens:9,.0f} tokens "
          f"-> {max_tokens / seq_tokens:6.1f} concurrent 4K sequences")
# v5e: 7 GB / 73,728 B = ~94,924 tokens -> ~23 sequences. v6e: ~76. H100: ~232.
assert abs((16 - 8 - 1) * 1e9 / kv_per_token / seq_tokens - 23.2) < 0.2
assert abs((32 - 8 - 1) * 1e9 / kv_per_token / seq_tokens - 76.2) < 0.2
assert abs((80 - 8 - 2) * 1e9 / kv_per_token / seq_tokens - 231.8) < 0.2

In [ ]:
# 4. Compile-cache break-even: first boot compiles graphs ("a few minutes"),
# VLLM_XLA_CACHE_PATH reuses them. 12 distinct shapes x 25 s each = 5 min first boot;
# cached restart ~= model load, 45 s. Two restarts/day for a month.
shapes, compile_each_s = 12, 25
first_boot_s = shapes * compile_each_s
cached_s = 45
restarts_per_day, days = 2, 30
uncached = restarts_per_day * days * first_boot_s / 60
cached = restarts_per_day * days * cached_s / 60
print(f"First boot compile: {first_boot_s/60:.0f} min; cached restart: {cached_s} s")
print(f"30 days, 2 restarts/day — uncached: {uncached:.0f} min, cached: {cached:.0f} min, "
      f"saved: {uncached - cached:.0f} min ({(uncached - cached)/60:.2f} hr)")
assert first_boot_s == 300 and uncached - cached == 255.0